# GAN и генерация изображений - как нейросети учатся создавать

**Роль:** Преподаватель по ML
**Уровень:** средний (основы PyTorch + CNN)
**Время прохождения:** ~120-150 минут

---

### Чему вы научитесь

После прохождения этого блокнота вы:
- поймёте, **что такое GAN** и почему состязательное обучение работает;
- разберёте **минимаксную игру** между Генератором и Дискриминатором;
- реализуете **простой GAN с нуля** на 1D данных (синусоиды);
- визуализируете, **как** GAN учится шаг за шагом;
- построите **DCGAN** для генерации изображений на MNIST;
- исследуете **латентное пространство** через интерактивные виджеты;
- создадите **FastAPI сервер** для генерации изображений;
- реализуете **условный GAN** (cGAN) для генерации конкретных цифр;
- диагностируете **mode collapse** и проблемы нестабильности;
- сравните **GAN, VAE, Diffusion** модели.

### Принцип этого блокнота

> Вы - автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Что такое GAN? | Генератор vs Дискриминатор, состязательная игра |
| 3 | Математика GAN: минимакс | Функция потерь, равновесие Нэша |
| 4 | Простой GAN с нуля | Генератор + Дискриминатор для 1D данных |
| 5 | Визуализация обучения | Пошаговое сравнение распределений |
| 6 | DCGAN: глубоко-свёрточный GAN | Архитектура для генерации изображений |
| 7 | Интерактивный эксплорер латентного пространства | Слайдеры ipywidgets для латентного вектора |
| 8 | FastAPI: сервер генерации изображений | Эндпоинты /generate, /interpolate, ngrok |
| 9 | Условный GAN (cGAN) | Генерация конкретных цифр |
| 10 | Проблемы обучения GAN | Mode collapse, нестабильность, решения |
| 11 | Концепция StyleGAN | Прогрессивный рост, инъекция стиля |
| 12 | Не только GAN: VAE, Diffusion | Сравнительная таблица |
| 13 | Практические задания | 5 упражнений |

---

---
## Шаг 1. Подготовка окружения - установка и импорт библиотек

| Библиотека | Зачем |
|-----------|-------|
| **torch** | Создание и обучение GAN моделей |
| **torchvision** | Датасет MNIST и трансформации изображений |
| **matplotlib** | Визуализация: графики, сгенерированные изображения |
| **ipywidgets** | Интерактивные виджеты для исследования латентного пространства |
| **fastapi** | Создание сервера для генерации изображений |
| **uvicorn** | ASGI-сервер для FastAPI |
| **pyngrok** | Публичный доступ к серверу из Colab |
| **numpy** | Математические операции и массивы

In [ ]:
# ============================================================
# ШАГ 1: Устанавливаем необходимые библиотеки
# ============================================================

!pip install -q fastapi uvicorn pyngrok               # библиотеки для сервера
!pip install -q torch torchvision                       # ML фреймворк (обычно уже есть в Colab)

print("Все библиотеки установлены!")

In [ ]:
# ============================================================
# ШАГ 1 (продолжение): Импортируем все нужные модули
# ============================================================

import numpy as np                                  # основная библиотека для массивов и математики
import matplotlib.pyplot as plt                     # для построения графиков и визуализаций
import torch                                        # PyTorch - основной фреймворк
import torch.nn as nn                               # слои нейросети (Linear, Conv2d...)
import torch.nn.functional as F                     # функции активации, функции потерь
import torch.optim as optim                         # оптимизаторы (Adam, SGD...)
from torch.utils.data import DataLoader, TensorDataset  # утилиты загрузки данных
import torchvision                                  # утилиты компьютерного зрения
from torchvision import datasets, transforms        # датасеты и трансформации изображений
import json                                         # для работы с форматом JSON
import os                                           # для операций с файловой системой
import io                                           # для операций с байтовыми потоками
import base64                                       # для кодирования изображений в base64
import time                                         # для замера времени операций
from IPython.display import display, HTML, clear_output  # красивый вывод в блокноте
import ipywidgets as widgets                        # интерактивные виджеты
from ipywidgets import interact, interactive, fixed, Layout  # декораторы виджетов

# Настраиваем matplotlib для лучшего отображения
plt.rcParams['figure.figsize'] = (12, 7)           # размер фигуры по умолчанию
plt.rcParams['font.size'] = 12                     # размер шрифта
plt.rcParams['axes.grid'] = True                   # показывать сетку

# Фиксируем случайные семена для воспроизводимости
torch.manual_seed(42)                              # семя для PyTorch
np.random.seed(42)                                 # семя для NumPy

# Определяем устройство: GPU если доступно, иначе CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU или CPU

print(f"PyTorch: {torch.__version__}")             # показываем версию PyTorch
print(f"Устройство: {device}")                     # показываем выбранное устройство
print("Все модули импортированы!")

---
## Шаг 2. Что такое GAN?

GAN (Generative Adversarial Network) - это фреймворк, в котором две нейросети соревнуются:

- **Генератор (G)** - создаёт фейковые данные, пытаясь обмануть дискриминатор
- **Дискриминатор (D)** - пытается отличить реальные данные от фейковых

### Аналогия: Фальшивомонетчик vs Полиция

| Роль | Компонент GAN | Цель |
|------|--------------|------|
| Фальшивомонетчик | Генератор | Создать купюры, которые выглядят как настоящие |
| Полиция | Дискриминатор | Обнаружить поддельные купюры |
| Игра | Состязательное обучение | Оба становятся лучше со временем |

### Процесс обучения

1. Генератор создаёт фейковые данные из случайного шума (латентный вектор z)
2. Дискриминатор оценивает: реальное или фейк?
3. Генератор получает награду, когда обманывает Дискриминатор
4. Дискриминатор получает награду, когда ловит фейки
5. Оба улучшаются до равновесия

```
Случайный шум z -> [Генератор] -> Фейковые данные x_fake
Реальные данные x_real ------------------------+
                                          v
                              [Дискриминатор] -> Реальное/Фейк?
```

В равновесии: Генератор создаёт данные неотличимые от реальных, Дискриминатор выдаёт 0.5 (случайное угадывание).

In [ ]:
# ============================================================
# ШАГ 2: Визуализируем архитектуру GAN
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(14, 6))     # создаём фигуру с одной осью

# Рисуем блок Генератора
gen_box = plt.Rectangle((0.05, 0.3), 0.25, 0.4,    # x, y, ширина, высота
                         facecolor='#4CAF50',       # зелёный цвет
                         edgecolor='black',          # чёрная рамка
                         linewidth=2, alpha=0.8)     # толщина рамки и прозрачность
ax.add_patch(gen_box)                                # добавляем блок генератора на график
ax.text(0.175, 0.5, 'Генератор\nG(z)',              # текст подписи
        ha='center', va='center',                     # центрирование
        fontsize=14, fontweight='bold', color='white')  # настройки шрифта

# Рисуем блок Дискриминатора
disc_box = plt.Rectangle((0.55, 0.3), 0.25, 0.4,   # x, y, ширина, высота
                          facecolor='#F44336',       # красный цвет
                          edgecolor='black',          # чёрная рамка
                          linewidth=2, alpha=0.8)     # толщина рамки и прозрачность
ax.add_patch(disc_box)                               # добавляем блок дискриминатора на график
ax.text(0.675, 0.5, 'Дискриминатор\nD(x)',          # текст подписи
        ha='center', va='center',                     # центрирование
        fontsize=14, fontweight='bold', color='white')  # настройки шрифта

# Стрелка: шум z -> Генератор
ax.annotate('', xy=(0.05, 0.5), xytext=(-0.05, 0.5),  # координаты стрелки
            arrowprops=dict(arrowstyle='->', lw=2.5, color='blue'))  # стиль стрелки
ax.text(-0.05, 0.62, 'Шум z',                       # подпись шума
        ha='center', fontsize=12, color='blue', fontweight='bold')  # стиль подписи

# Стрелка: Генератор -> Дискриминатор (фейк)
ax.annotate('', xy=(0.55, 0.55), xytext=(0.30, 0.55),  # координаты стрелки
            arrowprops=dict(arrowstyle='->', lw=2.5, color='orange'))  # стиль стрелки
ax.text(0.425, 0.68, 'Фейк x_fake',                    # подпись фейковых данных
        ha='center', fontsize=11, color='orange', fontweight='bold')  # стиль подписи

# Стрелка: Реальные данные -> Дискриминатор
ax.annotate('', xy=(0.55, 0.45), xytext=(0.42, 0.15),  # координаты стрелки
            arrowprops=dict(arrowstyle='->', lw=2.5, color='green'))  # стиль стрелки
ax.text(0.42, 0.08, 'Реальные x_real',                 # подпись реальных данных
        ha='center', fontsize=11, color='green', fontweight='bold')  # стиль подписи

# Стрелка: Дискриминатор -> выход
ax.annotate('', xy=(0.92, 0.5), xytext=(0.80, 0.5),    # координаты стрелки
            arrowprops=dict(arrowstyle='->', lw=2.5, color='purple'))  # стиль стрелки
ax.text(0.92, 0.58, 'Реальное/Фейк?',                    # подпись выхода
        ha='center', fontsize=12, color='purple', fontweight='bold')  # стиль подписи

# Стрелки обратной связи
ax.annotate('Потеря G:\nобмануть D', xy=(0.30, 0.32), xytext=(0.55, 0.25),  # обратная связь G
            arrowprops=dict(arrowstyle='->', lw=1.5, color='orange',    # стиль стрелки
                           connectionstyle='arc3,rad=0.3'),             # изогнутая стрелка
            fontsize=10, color='orange', ha='center')                   # стиль подписи

ax.annotate('Потеря D:\nпоймать фейки', xy=(0.55, 0.68), xytext=(0.30, 0.75),  # обратная связь D
            arrowprops=dict(arrowstyle='->', lw=1.5, color='red',       # стиль стрелки
                           connectionstyle='arc3,rad=0.3'),             # изогнутая стрелка
            fontsize=10, color='red', ha='center')                      # стиль подписи

ax.set_xlim(-0.12, 1.05)                            # пределы оси X
ax.set_ylim(0.0, 0.95)                              # пределы оси Y
ax.set_title('Архитектура GAN: Генератор vs Дискриминатор', fontsize=16, fontweight='bold')  # заголовок
ax.axis('off')                                       # скрываем деления осей

plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

---
## Шаг 3. Математика GAN: минимаксная игра

### Функция потерь GAN (Goodfellow, 2014)

Цель GAN - это **минимаксная игра**:

```
min_G max_D V(D, G) = E_x~p_data[log D(x)] + E_z~p_z[log(1 - D(G(z)))]
```

Разберём по частям:

| Слагаемое | Смысл | Кто оптимизирует |
|-----------|-------|-----------------|
| `E_x~p_data[log D(x)]` | Дискриминатор хочет D(x_real) -> 1 | D максимизирует |
| `E_z~p_z[log(1 - D(G(z)))]` | D хочет D(x_fake) -> 0, G хочет D(x_fake) -> 1 | D максимизирует, G минимизирует |

### Интуитивное объяснение

- **Цель Дискриминатора** (максимизировать): правильно классифицировать реальное как 1, фейк как 0
- **Цель Генератора** (минимизировать): сделать фейк таким хорошим, чтобы D выдавал 1 для фейков

### Равновесие Нэша

В глобальном оптимуме:
- p_G = p_data (Генератор идеально совпадает с реальным распределением)
- D(x) = 0.5 для всех x (Дискриминатор может только угадывать случайно)

### Обучение на практике

Вместо минимизации `log(1 - D(G(z)))`, Генератор часто максимизирует `log D(G(z))` для лучших градиентов на ранних этапах обучения.

In [ ]:
# ============================================================
# ШАГ 3: Визуализируем минимаксную игру на простом примере
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))    # создаём 3 подграфика

# Подграфик 1: Раннее обучение - D легко различает
x = np.linspace(-3, 3, 300)                        # значения оси X
p_real = 0.6 * np.exp(-0.5 * (x - 1)**2) + 0.4 * np.exp(-0.5 * (x + 1.5)**2)  # реальное распределение (бимодальное)
p_fake_early = 0.8 * np.exp(-0.5 * (x + 0.5)**2)  # фейковое распределение (раннее, далеко от реального)
d_early = p_real / (p_real + p_fake_early + 1e-8)  # оптимальный D = p_real / (p_real + p_fake)

axes[0].plot(x, p_real, 'b-', linewidth=2, label='p_data (реальное)')  # реальное распределение
axes[0].plot(x, p_fake_early, 'r--', linewidth=2, label='p_G (фейк)')  # фейковое распределение
axes[0].fill_between(x, p_real, alpha=0.2, color='blue')  # заливка реального
axes[0].fill_between(x, p_fake_early, alpha=0.2, color='red')  # заливка фейка
axes[0].set_title('Раннее обучение\nD легко различает реальное и фейк', fontsize=13)  # заголовок
axes[0].legend(fontsize=11)                         # показываем легенду
axes[0].set_xlabel('x')                             # подпись оси X
axes[0].set_ylabel('Плотность')                     # подпись оси Y

# Подграфик 2: Середина обучения - G улучшается
p_fake_mid = 0.5 * np.exp(-0.5 * (x - 0.8)**2) + 0.3 * np.exp(-0.5 * (x + 1.2)**2)  # приближается
d_mid = p_real / (p_real + p_fake_mid + 1e-8)      # оптимальный D в середине обучения

axes[1].plot(x, p_real, 'b-', linewidth=2, label='p_data (реальное)')  # реальное распределение
axes[1].plot(x, p_fake_mid, 'r--', linewidth=2, label='p_G (фейк)')  # фейковое распределение
axes[1].fill_between(x, p_real, alpha=0.2, color='blue')  # заливка реального
axes[1].fill_between(x, p_fake_mid, alpha=0.2, color='red')  # заливка фейка
axes[1].set_title('Середина обучения\nG приближается к реальному распределению', fontsize=13)  # заголовок
axes[1].legend(fontsize=11)                         # показываем легенду
axes[1].set_xlabel('x')                             # подпись оси X

# Подграфик 3: Равновесие - p_G = p_data, D = 0.5
p_fake_eq = p_real.copy()                           # в равновесии фейк = реальное
d_eq = p_real / (p_real + p_fake_eq + 1e-8)        # D = 0.5 везде

axes[2].plot(x, p_real, 'b-', linewidth=2, label='p_data (реальное)')  # реальное распределение
axes[2].plot(x, p_fake_eq, 'r--', linewidth=2, label='p_G (фейк)')  # фейк = реальное
axes[2].fill_between(x, p_real, alpha=0.2, color='blue')  # заливка реального
axes[2].fill_between(x, p_fake_eq, alpha=0.15, color='red')  # заливка фейка (перекрывается)
axes[2].set_title('Равновесие\np_G = p_data, D(x) = 0.5', fontsize=13)  # заголовок
axes[2].legend(fontsize=11)                         # показываем легенду
axes[2].set_xlabel('x')                             # подпись оси X

plt.suptitle('Минимаксная игра GAN: прогресс обучения', fontsize=15, fontweight='bold', y=1.02)  # общий заголовок
plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

In [ ]:
# ============================================================
# ШАГ 3 (продолжение): Визуализируем D(x) в процессе обучения
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))    # создаём 3 подграфика

x = np.linspace(-3, 3, 300)                        # значения оси X
p_real = 0.6 * np.exp(-0.5 * (x - 1)**2) + 0.4 * np.exp(-0.5 * (x + 1.5)**2)  # реальное распределение
p_fake_early = 0.8 * np.exp(-0.5 * (x + 0.5)**2)  # ранний фейк
p_fake_mid = 0.5 * np.exp(-0.5 * (x - 0.8)**2) + 0.3 * np.exp(-0.5 * (x + 1.2)**2)  # средний фейк
p_fake_eq = p_real.copy()                           # фейк в равновесии

# Вычисляем оптимальный D для каждого этапа
d_early = p_real / (p_real + p_fake_early + 1e-8)  # D на раннем этапе
d_mid = p_real / (p_real + p_fake_mid + 1e-8)      # D на среднем этапе
d_eq = p_real / (p_real + p_fake_eq + 1e-8)        # D в равновесии

# Строим D(x) для каждого этапа
for ax, d_val, title in [                           # перебираем подграфики
    (axes[0], d_early, 'Ранний этап: D уверен'),
    (axes[1], d_mid, 'Середина: D менее уверен'),
    (axes[2], d_eq, 'Равновесие: D = 0.5')]:
    ax.plot(x, d_val, 'purple', linewidth=2.5)      # строим D(x)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)  # опорная линия 0.5
    ax.fill_between(x, d_val, 0.5, alpha=0.2, color='purple')  # заштрихованная область
    ax.set_ylim(0, 1)                               # пределы оси Y
    ax.set_title(title, fontsize=13)                # заголовок
    ax.set_xlabel('x')                              # подпись оси X
    ax.set_ylabel('D(x)')                           # подпись оси Y

plt.suptitle('Оптимальный Дискриминатор D*(x) = p_data / (p_data + p_G)', fontsize=15, fontweight='bold', y=1.02)  # общий заголовок
plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

---
## Шаг 4. Простой GAN с нуля на PyTorch

Мы строим GAN для 1D данных: генератор учится создавать синусоиды из случайного шума.

### Наши данные
- Реальное распределение: смесь синусоид с разными фазами
- Генератор: MLP, отображающий z (латентный шум) -> x (1D выход)
- Дискриминатор: MLP, отображающий x -> вероятность быть реальным

### Архитектура

```
z (латентный, dim=8) -> [G: 8->32->32->1] -> x_fake (1D)
x (реальное или фейк) -> [D: 1->32->32->1] -> вероятность
```

In [ ]:
# ============================================================
# ШАГ 4: Определяем Генератор и Дискриминатор для 1D данных
# ============================================================

class Generator1D(nn.Module):                       # Генератор для 1D данных
    # Отображает латентный вектор z -> 1D выход x
    def __init__(self, latent_dim=8, hidden_dim=32):  # конструктор
        super(Generator1D, self).__init__()          # вызываем конструктор родителя
        self.latent_dim = latent_dim                 # сохраняем размерность латентного пространства
        self.net = nn.Sequential(                    # последовательная сеть
            nn.Linear(latent_dim, hidden_dim),       # первый слой: латентный -> скрытый
            nn.LeakyReLU(0.2),                      # leaky ReLU активация
            nn.Linear(hidden_dim, hidden_dim),       # второй скрытый слой
            nn.LeakyReLU(0.2),                      # leaky ReLU активация
            nn.Linear(hidden_dim, 1)                 # выходной слой: скрытый -> 1D
        )

    def forward(self, z):                           # прямой проход
        return self.net(z)                           # пропускаем через сеть


class Discriminator1D(nn.Module):                   # Дискриминатор для 1D данных
    # Отображает 1D вход x -> вероятность быть реальным
    def __init__(self, hidden_dim=32):              # конструктор
        super(Discriminator1D, self).__init__()      # вызываем конструктор родителя
        self.net = nn.Sequential(                    # последовательная сеть
            nn.Linear(1, hidden_dim),                # первый слой: 1D -> скрытый
            nn.LeakyReLU(0.2),                      # leaky ReLU активация
            nn.Linear(hidden_dim, hidden_dim),       # второй скрытый слой
            nn.LeakyReLU(0.2),                      # leaky ReLU активация
            nn.Linear(hidden_dim, 1),                # выходной слой: скрытый -> 1
            nn.Sigmoid()                             # сигмоида для вероятности [0,1]
        )

    def forward(self, x):                           # прямой проход
        return self.net(x)                           # пропускаем через сеть


# Создаём экземпляры моделей
latent_dim_1d = 8                                   # размерность латентного вектора
G1 = Generator1D(latent_dim=latent_dim_1d).to(device)  # генератор на устройстве
D1 = Discriminator1D().to(device)                   # дискриминатор на устройстве

print(f"Параметры Генератора: {sum(p.numel() for p in G1.parameters())}")  # считаем параметры G
print(f"Параметры Дискриминатора: {sum(p.numel() for p in D1.parameters())}")  # считаем параметры D
print("1D GAN модели созданы!")

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Генерируем реальное распределение данных
# ============================================================

def generate_real_data_1d(n_samples=256):           # генерируем реальные 1D примеры
    # Реальное распределение: смесь синусоидальных паттернов + шум
    t = torch.linspace(0, 4 * np.pi, n_samples)    # значения времени для синуса
    phase = np.random.choice([0, np.pi/2, np.pi, 3*np.pi/2])  # случайная фаза
    data = torch.sin(t + phase).unsqueeze(1)        # синусоида с фазой
    noise = torch.randn_like(data) * 0.3            # добавляем гауссов шум
    return (data + noise).to(device)                 # возвращаем зашумлённую синусоиду

# Визуализируем реальное распределение данных
real_samples = generate_real_data_1d(512)           # генерируем 512 примеров

fig, axes = plt.subplots(1, 2, figsize=(14, 5))    # создаём 2 подграфика

# Строим синусоиду
axes[0].plot(real_samples[:100, 0].cpu().numpy(), 'b-', alpha=0.5, linewidth=0.8)  # первые 100 примеров
axes[0].set_title('Реальные данные: примеры синусоид', fontsize=13)  # заголовок
axes[0].set_xlabel('Индекс примера')                  # подпись оси X
axes[0].set_ylabel('Значение')                         # подпись оси Y

# Строим гистограмму значений
axes[1].hist(real_samples[:, 0].cpu().numpy(), bins=50, color='blue', alpha=0.7, density=True)  # гистограмма
axes[1].set_title('Реальные данные: распределение значений', fontsize=13)  # заголовок
axes[1].set_xlabel('Значение')                         # подпись оси X
axes[1].set_ylabel('Плотность')                       # подпись оси Y

plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Цикл обучения 1D GAN
# ============================================================

# Оптимизаторы
lr_1d = 0.0002                                     # скорость обучения
optimizer_G1 = optim.Adam(G1.parameters(), lr=lr_1d, betas=(0.5, 0.999))  # Adam для Генератора
optimizer_D1 = optim.Adam(D1.parameters(), lr=lr_1d, betas=(0.5, 0.999))  # Adam для Дискриминатора

# Функция потерь
criterion = nn.BCELoss()                            # бинарная кросс-энтропия

# Параметры обучения
num_epochs_1d = 300                                 # количество эпох обучения
batch_size_1d = 128                                 # размер батча

# Хранилище потерь
g_losses_1d = []                                    # история потерь генератора
d_losses_1d = []                                    # история потерь дискриминатора

# Снимки сгенерированных данных на определённых эпохах
snapshots = {}                                      # словарь для хранения снимков
snapshot_epochs = [1, 50, 100, 200, 300]            # эпохи для сохранения снимков

for epoch in range(1, num_epochs_1d + 1):           # цикл обучения
    # Генерируем реальные и фейковые данные
    real_data = generate_real_data_1d(batch_size_1d)  # реальные примеры
    z = torch.randn(batch_size_1d, latent_dim_1d).to(device)  # случайные латентные векторы
    fake_data = G1(z).detach()                      # генерируем фейк (detach для обучения D)

    # --- Обучаем Дискриминатор ---
    optimizer_D1.zero_grad()                        # обнуляем градиенты D

    # Потеря D на реальных данных: D должен выдавать 1 для реальных
    real_labels = torch.ones(batch_size_1d, 1).to(device)  # метки реальных = 1
    d_real_loss = criterion(D1(real_data), real_labels)  # потеря на реальных данных

    # Потеря D на фейковых данных: D должен выдавать 0 для фейков
    fake_labels = torch.zeros(batch_size_1d, 1).to(device)  # метки фейков = 0
    d_fake_loss = criterion(D1(fake_data), fake_labels)  # потеря на фейковых данных

    d_loss = (d_real_loss + d_fake_loss) / 2        # средняя потеря D
    d_loss.backward()                                # обратное распространение
    optimizer_D1.step()                              # обновляем веса D

    # --- Обучаем Генератор ---
    optimizer_G1.zero_grad()                        # обнуляем градиенты G

    z = torch.randn(batch_size_1d, latent_dim_1d).to(device)  # новые случайные латентные векторы
    fake_data_g = G1(z)                             # генерируем фейк (с градиентами)
    g_loss = criterion(D1(fake_data_g), real_labels)  # G хочет, чтобы D выдавал 1 для фейков
    g_loss.backward()                                # обратное распространение
    optimizer_G1.step()                              # обновляем веса G

    # Записываем потери
    g_losses_1d.append(g_loss.item())               # сохраняем потерю G
    d_losses_1d.append(d_loss.item())               # сохраняем потерю D

    # Сохраняем снимок на определённых эпохах
    if epoch in snapshot_epochs:                    # проверяем нужен ли снимок
        with torch.no_grad():                       # без градиентов для оценки
            z_snap = torch.randn(512, latent_dim_1d).to(device)  # латентные для снимка
            snapshots[epoch] = G1(z_snap).cpu().numpy()  # сохраняем сгенерированные данные

    # Печатаем прогресс
    if epoch % 50 == 0 or epoch == 1:              # печатаем каждые 50 эпох
        print(f"Эпоха {epoch:4d}/{num_epochs_1d} | Потеря D: {d_loss.item():.4f} | Потеря G: {g_loss.item():.4f}")  # прогресс

print("\nОбучение 1D GAN завершено!")

---
## Шаг 5. Визуализация обучения: как GAN учится

Мы сравниваем сгенерированное распределение с реальным на разных этапах обучения.
Это показывает, как Генератор постепенно учится совпадать с реальным распределением данных.

In [ ]:
# ============================================================
# ШАГ 5: Визуализируем потери обучения по эпохам
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))    # создаём 2 подграфика

# Строим потери G и D
axes[0].plot(g_losses_1d, label='Потеря Генератора', color='orange', alpha=0.8)  # потеря G
axes[0].plot(d_losses_1d, label='Потеря Дискриминатора', color='blue', alpha=0.8)  # потеря D
axes[0].set_title('Потери обучения GAN (1D)', fontsize=14)  # заголовок
axes[0].set_xlabel('Эпоха')                         # подпись оси X
axes[0].set_ylabel('Потеря')                          # подпись оси Y
axes[0].legend(fontsize=12)                         # показываем легенду

# Сглаженные потери (скользящее среднее)
window = 20                                         # окно сглаживания
g_smooth = np.convolve(g_losses_1d, np.ones(window)/window, mode='valid')  # сглаженная потеря G
d_smooth = np.convolve(d_losses_1d, np.ones(window)/window, mode='valid')  # сглаженная потеря D
axes[1].plot(g_smooth, label='Генератор (сглаж.)', color='orange', linewidth=2)  # сглаженная потеря G
axes[1].plot(d_smooth, label='Дискриминатор (сглаж.)', color='blue', linewidth=2)  # сглаженная потеря D
axes[1].set_title('Сглаженные потери обучения', fontsize=14)  # заголовок
axes[1].set_xlabel('Эпоха')                         # подпись оси X
axes[1].set_ylabel('Потеря')                          # подпись оси Y
axes[1].legend(fontsize=12)                         # показываем легенду

plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

In [ ]:
# ============================================================
# ШАГ 5 (продолжение): Визуализируем сгенерированное vs реальное
# распределение на разных этапах обучения
# ============================================================

fig, axes = plt.subplots(1, len(snapshot_epochs), figsize=(20, 4))  # по подграфику на каждую эпоху

real_vals = generate_real_data_1d(512).cpu().numpy()  # реальные данные для сравнения

for i, epoch in enumerate(snapshot_epochs):         # перебираем эпохи снимков
    if epoch in snapshots:                           # проверяем существование снимка
        fake_vals = snapshots[epoch]                 # получаем снимок фейковых данных
        axes[i].hist(real_vals[:, 0], bins=50, alpha=0.5, color='blue',  # гистограмма реальных
                     label='Реальное', density=True)
        axes[i].hist(fake_vals[:, 0], bins=50, alpha=0.5, color='red',   # гистограмма фейковых
                     label='Сгенерированное', density=True)
        axes[i].set_title(f'Эпоха {epoch}', fontsize=12)  # заголовок с номером эпохи
        axes[i].set_xlabel('Значение')                  # подпись оси X
        if i == 0:                                   # только первый подграфик получает подпись Y
            axes[i].set_ylabel('Плотность')            # подпись оси Y
        axes[i].legend(fontsize=9)                   # показываем легенду

plt.suptitle('1D GAN: Сгенерированное vs Реальное распределение на разных эпохах', fontsize=14, fontweight='bold')  # общий заголовок
plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

In [ ]:
# ============================================================
# ШАГ 5 (продолжение): Визуализация формы синусоиды на разных эпохах
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))   # сетка 2x3 подграфиков
axes = axes.flatten()                               # выравниваем для удобной индексации

# Показываем сгенерированные синусоиды на каждой эпохе снимка
for i, epoch in enumerate(snapshot_epochs):         # перебираем эпохи
    if epoch in snapshots and i < len(axes):        # проверяем границы
        fake_vals = snapshots[epoch]                 # получаем снимок
        axes[i].plot(fake_vals[:80, 0], 'r-', alpha=0.7, linewidth=1.5, label='Сгенерированное')  # сгенерированная волна
        axes[i].plot(real_vals[:80, 0], 'b-', alpha=0.4, linewidth=1, label='Реальное')  # реальная волна
        axes[i].set_title(f'Эпоха {epoch}', fontsize=13)  # заголовок
        axes[i].legend(fontsize=10)                  # легенда
        axes[i].set_xlabel('Индекс примера')           # подпись оси X

# Скрываем лишние подграфики
if len(snapshot_epochs) < len(axes):                # если есть лишние подграфики
    for j in range(len(snapshot_epochs), len(axes)):  # скрываем оставшиеся
        axes[j].axis('off')                          # скрываем ось

plt.suptitle('Сгенерированные синусоиды на разных этапах обучения', fontsize=15, fontweight='bold')  # заголовок
plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

---
## Шаг 6. DCGAN: глубоко-свёрточный GAN

DCGAN (Deep Convolutional GAN) использует свёрточные слои для генерации изображений.

### Ключевые архитектурные принципы (Radford et al., 2015)

| Принцип | Описание |
|---------|----------|
| Замена пулинга на свёртки с шагом | D использует Conv2d с stride, G использует ConvTranspose2d |
| Использование BatchNorm | В обоих G и D (кроме выхода G и входа D) |
| Убираем полносвязные слои | Глобальный пулинг в D, прямые свёртки в G |
| Используем ReLU в G | Кроме выхода: Tanh |
| Используем LeakyReLU в D | Наклон 0.2 |

### Архитектура Генератора для MNIST (28x28)

```
z (100) -> ConvTranspose2d(100,256,4,1,0) -> 256x4x4
        -> ConvTranspose2d(256,128,3,2,1) -> 128x7x7
        -> ConvTranspose2d(128,64,4,2,1)  -> 64x14x14
        -> ConvTranspose2d(64,1,4,2,1)    -> 1x28x28
```

In [ ]:
# ============================================================
# ШАГ 6: Определяем DCGAN Генератор для MNIST
# ============================================================

class DCGANGenerator(nn.Module):                    # DCGAN Генератор
    # Трансформирует латентный вектор z -> 28x28 серое изображение
    def __init__(self, latent_dim=100, ngf=64):     # конструктор
        super(DCGANGenerator, self).__init__()      # вызываем конструктор родителя
        self.latent_dim = latent_dim                 # сохраняем размерность латентного пространства
        self.main = nn.Sequential(                   # последовательная сеть
            # Вход: z (latent_dim x 1 x 1)
            nn.ConvTranspose2d(latent_dim, ngf * 4, 4, 1, 0, bias=False),  # -> 256x4x4
            nn.BatchNorm2d(ngf * 4),                 # пакетная нормализация
            nn.ReLU(True),                           # ReLU активация

            # Состояние: (ngf*4) x 4 x 4
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 3, 2, 1, bias=False),  # -> 128x7x7
            nn.BatchNorm2d(ngf * 2),                 # пакетная нормализация
            nn.ReLU(True),                           # ReLU активация

            # Состояние: (ngf*2) x 7 x 7
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),  # -> 64x14x14
            nn.BatchNorm2d(ngf),                     # пакетная нормализация
            nn.ReLU(True),                           # ReLU активация

            # Состояние: ngf x 14 x 14
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False),  # -> 1x28x28
            nn.Tanh()                                # Tanh: выход в [-1, 1]
        )

    def forward(self, z):                           # прямой проход
        # Переформируем z в (batch, latent_dim, 1, 1) для ConvTranspose2d
        z = z.view(z.size(0), self.latent_dim, 1, 1)  # переформируем в 4D тензор
        return self.main(z)                          # пропускаем через сеть


class DCGANDiscriminator(nn.Module):                # DCGAN Дискриминатор
    # Классифицирует 28x28 изображение -> вероятность реальное/фейк
    def __init__(self, ndf=64):                     # конструктор
        super(DCGANDiscriminator, self).__init__()   # вызываем конструктор родителя
        self.main = nn.Sequential(                   # последовательная сеть
            # Вход: 1 x 28 x 28
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False),  # -> 64x14x14
            nn.LeakyReLU(0.2, inplace=True),        # leaky ReLU

            # Состояние: ndf x 14 x 14
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),  # -> 128x7x7
            nn.BatchNorm2d(ndf * 2),                 # пакетная нормализация
            nn.LeakyReLU(0.2, inplace=True),        # leaky ReLU

            # Состояние: (ndf*2) x 7 x 7
            nn.Conv2d(ndf * 2, ndf * 4, 3, 2, 1, bias=False),  # -> 256x4x4
            nn.BatchNorm2d(ndf * 4),                 # пакетная нормализация
            nn.LeakyReLU(0.2, inplace=True),        # leaky ReLU

            # Состояние: (ndf*4) x 4 x 4
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),  # -> 1x1x1
            nn.Sigmoid()                             # сигмоида для вероятности
        )

    def forward(self, x):                           # прямой проход
        output = self.main(x)                        # пропускаем через сеть
        return output.view(-1, 1)                    # выравниваем в (batch, 1)


# Создаём модели DCGAN
latent_dim_dc = 100                                 # размерность латентного пространства для DCGAN
G_dc = DCGANGenerator(latent_dim=latent_dim_dc).to(device)  # генератор на устройстве
D_dc = DCGANDiscriminator().to(device)              # дискриминатор на устройстве

print(f"Параметры Генератора DCGAN: {sum(p.numel() for p in G_dc.parameters()):,}")  # считаем параметры G
print(f"Параметры Дискриминатора DCGAN: {sum(p.numel() for p in D_dc.parameters()):,}")  # считаем параметры D

# Тестируем со случайным входом
z_test = torch.randn(1, latent_dim_dc).to(device)   # случайный латентный вектор
fake_img = G_dc(z_test)                             # генерируем тестовое изображение
print(f"Форма сгенерированного изображения: {fake_img.shape}")    # должно быть [1, 1, 28, 28]

# Тестируем дискриминатор
d_test = D_dc(fake_img)                             # выход дискриминатора
print(f"Выход Дискриминатора: {d_test.item():.4f}")  # должно быть около 0.5
print("Модели DCGAN созданы и протестированы!")

---
## Шаг 7. Обучение DCGAN на MNIST

Полный цикл обучения с визуализацией прогресса каждые N эпох.
Обучаем умеренное количество эпох, чтобы увидеть результаты за разумное время в Colab.

In [ ]:
# ============================================================
# ШАГ 7: Загружаем датасет MNIST
# ============================================================

# Скачиваем MNIST и применяем трансформации
transform_mnist = transforms.Compose([              # композиция трансформаций
    transforms.ToTensor(),                           # конвертируем в тензор [0,1]
    transforms.Normalize([0.5], [0.5])              # нормализуем к [-1, 1] (совпадает с Tanh)
])

mnist_dataset = datasets.MNIST(                     # загружаем датасет MNIST
    root='./data',                                  # директория данных
    train=True,                                     # используем обучающую выборку
    download=True,                                  # скачиваем если нет
    transform=transform_mnist                       # применяем трансформации
)

# Создаём загрузчик данных
batch_size_dc = 128                                 # размер батча для DCGAN
mnist_loader = DataLoader(                          # загрузчик данных
    mnist_dataset,                                  # датасет
    batch_size=batch_size_dc,                       # размер батча
    shuffle=True,                                   # перемешиваем данные
    num_workers=2                                   # параллельная загрузка данных
)

print(f"Размер датасета MNIST: {len(mnist_dataset)}")  # показываем размер датасета
print(f"Количество батчей: {len(mnist_loader)}")    # показываем количество батчей

# Визуализируем несколько реальных примеров MNIST
real_batch = next(iter(mnist_loader))               # получаем первый батч
fig, axes = plt.subplots(1, 8, figsize=(16, 2))    # 8 примеров изображений
for i in range(8):                                  # показываем 8 изображений
    axes[i].imshow(real_batch[0][i, 0].numpy(), cmap='gray')  # отображаем изображение
    axes[i].axis('off')                             # скрываем оси
    axes[i].set_title(f'Метка: {real_batch[1][i].item()}', fontsize=9)  # показываем метку
plt.suptitle('Реальные примеры MNIST', fontsize=13)     # заголовок
plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

In [ ]:
# ============================================================
# ШАГ 7 (продолжение): Инициализация весов по правилам DCGAN
# ============================================================

def weights_init(m):                                # функция инициализации весов
    # Инициализируем веса следуя рекомендациям DCGAN
    classname = m.__class__.__name__                 # получаем имя класса слоя
    if classname.find('Conv') != -1:                # если свёрточный слой
        nn.init.normal_(m.weight.data, 0.0, 0.02)  # нормальное распределение, среднее=0, стд=0.02
    elif classname.find('BatchNorm') != -1:         # если слой пакетной нормализации
        nn.init.normal_(m.weight.data, 1.0, 0.02)  # нормальное распределение, среднее=1, стд=0.02
        nn.init.constant_(m.bias.data, 0)           # смещение = 0

# Применяем инициализацию весов к обеим моделям
G_dc.apply(weights_init)                            # инициализируем веса Генератора
D_dc.apply(weights_init)                            # инициализируем веса Дискриминатора

print("Веса инициализированы по правилам DCGAN!")

In [ ]:
# ============================================================
# ШАГ 7 (продолжение): Обучаем DCGAN на MNIST
# ============================================================

# Гиперпараметры обучения
num_epochs_dc = 15                                  # количество эпох обучения
lr_dc = 0.0002                                     # скорость обучения
beta1 = 0.5                                        # Adam beta1

# Оптимизаторы
optimizer_G_dc = optim.Adam(G_dc.parameters(), lr=lr_dc, betas=(beta1, 0.999))  # оптимизатор G
optimizer_D_dc = optim.Adam(D_dc.parameters(), lr=lr_dc, betas=(beta1, 0.999))  # оптимизатор D

# Функция потерь
criterion_dc = nn.BCELoss()                         # бинарная кросс-энтропия

# Хранилище статистики обучения
g_losses_dc = []                                    # история потерь G
d_losses_dc = []                                    # история потерь D
d_real_acc = []                                     # точность D на реальных данных
d_fake_acc = []                                     # точность D на фейковых данных

# Фиксированные латентные векторы для отслеживания качества генерации
fixed_noise = torch.randn(64, latent_dim_dc, device=device)  # фиксированный z для стабильной визуализации

# Сетки сгенерированных изображений на разных эпохах
generated_grids = {}                                # словарь для хранения сеток

print(f"Начинаем обучение DCGAN на {num_epochs_dc} эпох...")
print(f"Датасет: MNIST ({len(mnist_dataset)} изображений)")
print(f"Размер батча: {batch_size_dc}")
print(f"Размерность латентного пространства: {latent_dim_dc}")
print("-" * 60)

for epoch in range(1, num_epochs_dc + 1):           # цикл по эпохам
    g_loss_epoch = 0.0                              # накапливаем потерю G
    d_loss_epoch = 0.0                              # накапливаем потерю D
    d_real_correct = 0                              # счётчик правильных реальных предсказаний
    d_fake_correct = 0                              # счётчик правильных фейковых предсказаний
    total_samples = 0                               # всего примеров обработано

    for i, (real_imgs, _) in enumerate(mnist_loader):  # цикл по батчам
        batch_size_curr = real_imgs.size(0)          # текущий размер батча
        real_imgs = real_imgs.to(device)             # переносим на устройство

        # Метки
        real_labels = torch.ones(batch_size_curr, 1, device=device)  # реальное = 1
        fake_labels = torch.zeros(batch_size_curr, 1, device=device)  # фейк = 0

        # --- Обучаем Дискриминатор ---
        optimizer_D_dc.zero_grad()                  # обнуляем градиенты D

        # D на реальных изображениях
        d_real_output = D_dc(real_imgs)              # предсказание D на реальных
        d_loss_real = criterion_dc(d_real_output, real_labels)  # потеря на реальных

        # D на фейковых изображениях
        z = torch.randn(batch_size_curr, latent_dim_dc, device=device)  # случайный шум
        fake_imgs = G_dc(z).detach()                 # генерируем фейки (detach для D)
        d_fake_output = D_dc(fake_imgs)              # предсказание D на фейках
        d_loss_fake = criterion_dc(d_fake_output, fake_labels)  # потеря на фейках

        d_loss = (d_loss_real + d_loss_fake) / 2     # общая потеря D
        d_loss.backward()                            # обратное распространение
        optimizer_D_dc.step()                        # обновляем веса D

        # Отслеживаем точность D
        d_real_correct += (d_real_output > 0.5).sum().item()  # правильные реальные предсказания
        d_fake_correct += (d_fake_output < 0.5).sum().item()  # правильные фейковые предсказания
        total_samples += batch_size_curr             # считаем примеры

        # --- Обучаем Генератор ---
        optimizer_G_dc.zero_grad()                  # обнуляем градиенты G

        z = torch.randn(batch_size_curr, latent_dim_dc, device=device)  # новый случайный шум
        fake_imgs_g = G_dc(z)                        # генерируем фейки с градиентами
        d_fake_output_g = D_dc(fake_imgs_g)          # предсказание D на фейках G
        g_loss = criterion_dc(d_fake_output_g, real_labels)  # G хочет чтобы D сказал "реальное"
        g_loss.backward()                            # обратное распространение
        optimizer_G_dc.step()                        # обновляем веса G

        # Накапливаем потери
        g_loss_epoch += g_loss.item()                # добавляем потерю G
        d_loss_epoch += d_loss.item()                # добавляем потерю D

    # Средние потери за эпоху
    g_losses_dc.append(g_loss_epoch / len(mnist_loader))  # средняя потеря G
    d_losses_dc.append(d_loss_epoch / len(mnist_loader))  # средняя потеря D
    d_real_acc.append(d_real_correct / total_samples)  # точность D на реальных
    d_fake_acc.append(d_fake_correct / total_samples)  # точность D на фейках

    # Сохраняем сгенерированные изображения на этой эпохе
    with torch.no_grad():                           # без градиентов для оценки
        fake_sample = G_dc(fixed_noise).cpu()        # генерируем из фиксированного шума
    generated_grids[epoch] = fake_sample             # сохраняем сетку генераций

    # Печатаем прогресс
    print(f"Эпоха [{epoch:2d}/{num_epochs_dc}] | "
          f"Потеря D: {d_losses_dc[-1]:.4f} | Потеря G: {g_losses_dc[-1]:.4f} | "
          f"D_реальное: {d_real_acc[-1]:.3f} | D_фейк: {d_fake_acc[-1]:.3f}")

print("\nОбучение DCGAN завершено!")

In [ ]:
# ============================================================
# ШАГ 7 (продолжение): Визуализируем прогресс обучения DCGAN
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))   # сетка 2x2

# Строим потери G и D
axes[0, 0].plot(g_losses_dc, label='Потеря Генератора', color='orange', linewidth=2)  # потеря G
axes[0, 0].plot(d_losses_dc, label='Потеря Дискриминатора', color='blue', linewidth=2)  # потеря D
axes[0, 0].set_title('Потери обучения DCGAN', fontsize=13)  # заголовок
axes[0, 0].set_xlabel('Эпоха')                      # подпись оси X
axes[0, 0].set_ylabel('Потеря')                       # подпись оси Y
axes[0, 0].legend(fontsize=11)                      # легенда

# Строим точность D
axes[0, 1].plot(d_real_acc, label='Точность D на реальных', color='green', linewidth=2)  # точность на реальных
axes[0, 1].plot(d_fake_acc, label='Точность D на фейках', color='red', linewidth=2)  # точность на фейках
axes[0, 1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Случайное угадывание')  # опорная линия
axes[0, 1].set_title('Точность Дискриминатора', fontsize=13)  # заголовок
axes[0, 1].set_xlabel('Эпоха')                      # подпись оси X
axes[0, 1].set_ylabel('Точность')                   # подпись оси Y
axes[0, 1].legend(fontsize=11)                      # легенда

# Показываем сгенерированные изображения на эпохе 1
if 1 in generated_grids:                            # проверяем существование эпохи 1
    grid_early = generated_grids[1]                  # получаем ранние изображения
    img_grid = np.zeros((28 * 8, 28 * 8))           # создаём пустую сетку
    for idx in range(min(64, grid_early.size(0))):   # заполняем сетку изображениями
        row = idx // 8                               # индекс строки
        col = idx % 8                                # индекс столбца
        img_grid[row*28:(row+1)*28, col*28:(col+1)*28] = grid_early[idx, 0].numpy()  # размещаем изображение
    axes[1, 0].imshow(img_grid, cmap='gray', vmin=-1, vmax=1)  # показываем сетку
    axes[1, 0].set_title('Сгенерированные изображения (Эпоха 1)', fontsize=13)  # заголовок
    axes[1, 0].axis('off')                          # скрываем оси

# Показываем сгенерированные изображения на последней эпохе
last_epoch = max(generated_grids.keys())            # получаем последнюю эпоху
grid_late = generated_grids[last_epoch]              # получаем поздние изображения
img_grid_late = np.zeros((28 * 8, 28 * 8))          # создаём пустую сетку
for idx in range(min(64, grid_late.size(0))):       # заполняем сетку изображениями
    row = idx // 8                                   # индекс строки
    col = idx % 8                                    # индекс столбца
    img_grid_late[row*28:(row+1)*28, col*28:(col+1)*28] = grid_late[idx, 0].numpy()  # размещаем изображение
axes[1, 1].imshow(img_grid_late, cmap='gray', vmin=-1, vmax=1)  # показываем сетку
axes[1, 1].set_title(f'Сгенерированные изображения (Эпоха {last_epoch})', fontsize=13)  # заголовок
axes[1, 1].axis('off')                              # скрываем оси

plt.suptitle('Прогресс обучения DCGAN на MNIST', fontsize=15, fontweight='bold')  # общий заголовок
plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

---
## Шаг 8. Интерактивный эксплорер латентного пространства

Используйте слайдеры ipywidgets для исследования того, как каждое измерение латентного вектора влияет на сгенерированное изображение.

**Как это работает:**
- Каждый слайдер управляет одним измерением 100-мерного латентного вектора z
- Перемещение слайдера изменяет сгенерированное изображение в реальном времени
- Кнопка "Случайный z" генерирует полностью случайный латентный вектор
- Слайдер интерполяции плавно переходит между двумя латентными точками

In [ ]:
# ============================================================
# ШАГ 8: Интерактивный эксплорер латентного пространства с ipywidgets
# ============================================================

# Используем только 8 регулируемых измерений (остальные фиксируем на 0)
# Это делает интерфейс управляемым, при этом показывая концепцию

n_sliders = 8                                      # количество регулируемых измерений

# Создаём слайдеры для каждого регулируемого латентного измерения
latent_sliders = []                                 # список для хранения слайдеров
for i in range(n_sliders):                          # создаём n слайдеров
    slider = widgets.FloatSlider(                   # виджет слайдера с плавающей точкой
        value=0.0,                                  # начальное значение
        min=-3.0,                                   # минимальное значение
        max=3.0,                                    # максимальное значение
        step=0.1,                                   # шаг
        description=f'z[{i}]',                      # подпись
        continuous_update=True,                     # обновлять при перетаскивании
        layout=Layout(width='350px')                # ширина слайдера
    )
    latent_sliders.append(slider)                   # добавляем в список

# Кнопка для генерации случайного латентного вектора
random_button = widgets.Button(                     # виджет кнопки
    description='Случайный z',                      # текст кнопки
    button_style='success',                         # зелёный стиль
    layout=Layout(width='150px')                    # ширина кнопки
)

# Область вывода для сгенерированного изображения
output_area = widgets.Output()                      # виджет вывода для отображения изображения

# Функция генерации изображения из значений слайдеров
def generate_from_sliders(slider_vals):             # функция генерации изображения
    z = torch.zeros(1, latent_dim_dc, device=device)  # начинаем со всех нулей
    for i, val in enumerate(slider_vals):           # устанавливаем регулируемые измерения
        z[0, i] = val                               # устанавливаем z[i] = значение слайдера
    with torch.no_grad():                           # без градиентов
        img = G_dc(z).cpu()                         # генерируем изображение
    return img[0, 0].numpy()                        # возвращаем как массив numpy

# Функция обновления для слайдеров
def update_image(change=None):                      # обновляем отображение при изменении слайдера
    vals = [s.value for s in latent_sliders]        # получаем все значения слайдеров
    img = generate_from_sliders(vals)               # генерируем изображение
    with output_area:                               # обновляем область вывода
        output_area.clear_output(wait=True)         # очищаем предыдущее изображение
        plt.figure(figsize=(3, 3))                  # маленькая фигура
        plt.imshow(img, cmap='gray', vmin=-1, vmax=1)  # показываем изображение
        plt.axis('off')                             # скрываем оси
        plt.title('Сгенерировано из z', fontsize=11)  # заголовок
        plt.show()                                   # отображаем

# Подключаем слайдеры к функции обновления
for slider in latent_sliders:                       # для каждого слайдера
    slider.observe(update_image, names='value')     # наблюдаем за изменениями

# Обработчик кнопки случайного z
def on_random_click(b):                             # обработчик кнопки случайного z
    for slider in latent_sliders:                   # для каждого слайдера
        slider.value = np.random.uniform(-2, 2)    # устанавливаем случайное значение
    update_image()                                  # обновляем отображение

random_button.on_click(on_random_click)             # подключаем кнопку

# Располагаем виджеты
slider_box = widgets.VBox(latent_sliders)           # вертикальный контейнер для слайдеров
control_box = widgets.HBox([slider_box, random_button, output_area])  # горизонтальная раскладка

display(control_box)                                # показываем виджет

# Начальное отображение
update_image()                                      # показываем начальное изображение

In [ ]:
# ============================================================
# ШАГ 8 (продолжение): Эксплорер интерполяции латентного пространства
# ============================================================

# Интерполяция между двумя случайными латентными векторами
z_start = torch.randn(1, latent_dim_dc, device=device)  # начальная точка z1
z_end = torch.randn(1, latent_dim_dc, device=device)    # конечная точка z2

# Слайдер интерполяции
interp_slider = widgets.FloatSlider(                # слайдер интерполяции
    value=0.0,                                      # начинаем с z_start
    min=0.0,                                        # минимум (z_start)
    max=1.0,                                        # максимум (z_end)
    step=0.01,                                      # шаг
    description='t',                                # подпись: параметр интерполяции
    continuous_update=True,                         # обновлять при перетаскивании
    layout=Layout(width='500px')                    # ширина слайдера
)

# Кнопки для новых случайных конечных точек
new_points_button = widgets.Button(                 # кнопка для новых точек
    description='Новые z1, z2',                     # текст кнопки
    button_style='info',                            # синий стиль
    layout=Layout(width='200px')                    # ширина кнопки
)

interp_output = widgets.Output()                    # область вывода для интерполяции

def update_interp(change=None):                     # обновляем отображение интерполяции
    t = interp_slider.value                         # получаем параметр интерполяции
    z_interp = (1 - t) * z_start + t * z_end       # линейная интерполяция: z = (1-t)*z1 + t*z2
    with torch.no_grad():                           # без градиентов
        img = G_dc(z_interp).cpu()                  # генерируем изображение в точке интерполяции
    with interp_output:                             # обновляем вывод
        interp_output.clear_output(wait=True)       # очищаем предыдущее
        plt.figure(figsize=(3, 3))                  # маленькая фигура
        plt.imshow(img[0, 0].numpy(), cmap='gray', vmin=-1, vmax=1)  # показываем изображение
        plt.axis('off')                             # скрываем оси
        plt.title(f't = {t:.2f}', fontsize=11)     # показываем значение t
        plt.show()                                   # отображаем

def on_new_points(b):                               # обработчик кнопки новых точек
    global z_start, z_end                           # модифицируем глобальные переменные
    z_start[:] = torch.randn(1, latent_dim_dc, device=device)  # новый z1
    z_end[:] = torch.randn(1, latent_dim_dc, device=device)    # новый z2
    interp_slider.value = 0.0                       # сбрасываем слайдер
    update_interp()                                  # обновляем отображение

interp_slider.observe(update_interp, names='value')  # наблюдаем за слайдером
new_points_button.on_click(on_new_points)           # подключаем кнопку

interp_box = widgets.HBox([interp_slider, new_points_button, interp_output])  # раскладка
display(interp_box)                                  # показываем виджет
update_interp()                                      # начальное отображение

In [ ]:
# ============================================================
# ШАГ 8 (продолжение): Полная сетка интерполяции латентного пространства
# Визуализируем плавный переход между z1 и z2
# ============================================================

def plot_interpolation_grid(n_steps=10):            # функция построения интерполяции
    z1 = torch.randn(1, latent_dim_dc, device=device)  # случайная начальная точка
    z2 = torch.randn(1, latent_dim_dc, device=device)  # случайная конечная точка

    fig, axes = plt.subplots(1, n_steps, figsize=(20, 2.5))  # ряд изображений

    for i in range(n_steps):                        # для каждого шага интерполяции
        t = i / (n_steps - 1)                       # параметр интерполяции [0, 1]
        z_t = (1 - t) * z1 + t * z2                 # линейная интерполяция
        with torch.no_grad():                       # без градиентов
            img = G_dc(z_t).cpu()                    # генерируем изображение
        axes[i].imshow(img[0, 0].numpy(), cmap='gray', vmin=-1, vmax=1)  # показываем изображение
        axes[i].axis('off')                         # скрываем оси
        axes[i].set_title(f't={t:.2f}', fontsize=9)  # показываем значение t

    plt.suptitle('Интерполяция латентного пространства: z1 -> z2', fontsize=14, fontweight='bold')  # заголовок
    plt.tight_layout()                               # подгоняем макет
    plt.show()                                       # отображаем фигуру

# Показываем две разные траектории интерполяции
print("Траектория интерполяции 1:")
plot_interpolation_grid(10)                          # первая интерполяция

print("Траектория интерполяции 2:")
plot_interpolation_grid(10)                          # вторая интерполяция

---
## Шаг 9. FastAPI сервер: API генерации изображений

Мы создаём FastAPI сервер со следующими эндпоинтами:

| Эндпоинт | Метод | Описание |
|----------|-------|----------|
| `/generate` | GET | Сгенерировать одно изображение из случайного или указанного латентного вектора |
| `/interpolate` | GET | Интерполировать между двумя латентными векторами |

Сервер использует ngrok для публичного доступа из Colab.

### Как использовать
1. Запустите ячейку с сервером ниже
2. Скопируйте ngrok URL
3. Используйте curl-команды для генерации изображений

In [ ]:
# ============================================================
# ШАГ 9: FastAPI сервер для генерации изображений
# ============================================================

from fastapi import FastAPI                          # фреймворк FastAPI
from fastapi.responses import StreamingResponse      # потоковый ответ для изображений
import uvicorn                                      # ASGI-сервер
from pyngrok import ngrok                           # ngrok для публичного доступа

app_gan = FastAPI(title="GAN API генерации изображений")  # создаём FastAPI приложение

# Вспомогательная функция: генерация изображения из латентного вектора
def generate_image_from_z(z_tensor):                # генерируем изображение из z
    with torch.no_grad():                           # градиенты не нужны
        img_tensor = G_dc(z_tensor).cpu()           # генерируем тензор изображения
    img_np = ((img_tensor[0, 0].numpy() + 1) / 2 * 255).astype(np.uint8)  # конвертируем в [0,255]
    return img_np                                    # возвращаем numpy изображение

# Вспомогательная функция: numpy изображение в PNG байты
def image_to_png_bytes(img_np):                     # конвертируем numpy в PNG байты
    buf = io.BytesIO()                              # байтовый буфер
    plt.imsave(buf, img_np, cmap='gray', format='png')  # сохраняем как PNG
    buf.seek(0)                                     # перематываем буфер
    return buf                                      # возвращаем буфер

# Вспомогательная функция: создаём сетку изображений
def create_image_grid(images_np, n_cols=4):         # создаём сетку из списка изображений
    n_images = len(images_np)                       # количество изображений
    n_rows = (n_images + n_cols - 1) // n_cols      # количество строк
    h, w = images_np[0].shape                       # размеры изображения
    grid = np.zeros((n_rows * h, n_cols * w), dtype=np.uint8)  # пустая сетка
    for idx, img in enumerate(images_np):           # размещаем каждое изображение
        row = idx // n_cols                          # индекс строки
        col = idx % n_cols                           # индекс столбца
        grid[row*h:(row+1)*h, col*w:(col+1)*w] = img  # размещаем изображение
    return grid                                      # возвращаем сетку

@app_gan.get("/")                                   # корневой эндпоинт
def read_root():                                    # обработчик корня
    return {                                        # информация об API
        "api": "GAN API генерации изображений",
        "endpoints": {
            "/generate": "Сгенерировать одно изображение (параметры: seed, z0..z7)",
            "/interpolate": "Интерполировать между двумя латентными векторами (параметры: steps)"
        }
    }

@app_gan.get("/generate")                           # генерация одного изображения
def generate_image(seed: int = None,                # опциональное семя для воспроизводимости
                   z0: float = None, z1: float = None, z2: float = None,  # опциональные измерения z
                   z3: float = None, z4: float = None, z5: float = None,
                   z6: float = None, z7: float = None):
    # Генерируем одно изображение из латентного вектора
    if seed is not None:                            # если предоставлено семя
        torch.manual_seed(seed)                     # устанавливаем семя для воспроизводимости
        z = torch.randn(1, latent_dim_dc, device=device)  # случайный z с семенем
    elif any(v is not None for v in [z0,z1,z2,z3,z4,z5,z6,z7]):  # если указаны какие-то измерения z
        z = torch.randn(1, latent_dim_dc, device=device) * 0.1  # начинаем с малого случайного
        for i, v in enumerate([z0,z1,z2,z3,z4,z5,z6,z7]):  # устанавливаем указанные измерения
            if v is not None:                       # если измерение было указано
                z[0, i] = v                         # устанавливаем это измерение
    else:                                           # нет семени, нет z
        z = torch.randn(1, latent_dim_dc, device=device)  # полностью случайный z

    img_np = generate_image_from_z(z)               # генерируем изображение
    buf = image_to_png_bytes(img_np)                # конвертируем в PNG байты
    return StreamingResponse(buf, media_type="image/png")  # возвращаем PNG изображение

@app_gan.get("/interpolate")                        # эндпоинт интерполяции
def interpolate_images(steps: int = 10):            # количество шагов интерполяции
    # Интерполируем между двумя случайными латентными векторами
    steps = min(max(2, steps), 20)                  # ограничиваем от 2 до 20 шагов
    z1 = torch.randn(1, latent_dim_dc, device=device)  # начальная точка
    z2 = torch.randn(1, latent_dim_dc, device=device)  # конечная точка
    images = []                                     # список изображений
    for i in range(steps):                          # для каждого шага
        t = i / (steps - 1)                         # параметр интерполяции
        z_t = (1 - t) * z1 + t * z2                 # линейная интерполяция
        img_np = generate_image_from_z(z_t)         # генерируем изображение
        images.append(img_np)                       # добавляем в список
    grid = create_image_grid(images, n_cols=steps)  # создаём однострочную сетку
    buf = image_to_png_bytes(grid)                  # конвертируем в PNG
    return StreamingResponse(buf, media_type="image/png")  # возвращаем сетку

print("FastAPI GAN сервер определён!")
print("Эндпоинты: /, /generate, /interpolate")

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Запускаем сервер с ngrok
# ============================================================

# Убиваем существующие ngrok туннели
ngrok.kill()                                        # убиваем существующие туннели

# Запускаем ngrok туннель
public_url = ngrok.connect(8000)                    # создаём публичный URL для порта 8000

print(f"Публичный URL: {public_url}")               # показываем публичный URL
print()
print("Тестовые команды (запустите в новой ячейке или терминале):")
print()
print(f"# 1. Сгенерировать случайное изображение:")
print(f"!curl -o generated.png '{public_url}/generate'")
print()
print(f"# 2. Сгенерировать изображение с семенем:")
print(f"!curl -o generated_seed42.png '{public_url}/generate?seed=42'")
print()
print(f"# 3. Сгенерировать изображение с конкретными латентными измерениями:")
print(f"!curl -o generated_custom.png '{public_url}/generate?z0=1.5&z1=-0.8&z2=2.0'")
print()
print(f"# 4. Интерполировать между двумя точками (10 шагов):")
print(f"!curl -o interpolation.png '{public_url}/interpolate?steps=10'")
print()
print("Сервер запускается... (эта ячейка будет работать, пока не прервёте)")

# Запускаем сервер
uvicorn.run(app_gan, host="0.0.0.0", port=8000)    # запускаем FastAPI сервер

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Тестируем GAN сервер (запустите ПОСЛЕ старта сервера)
# Скопируйте это в НОВУЮ ячейку и запустите пока сервер работает
# ============================================================

# ВАЖНО: Запустите это в отдельной ячейке, пока ячейка сервера ещё работает!
# Замените URL ниже на ваш ngrok URL из предыдущей ячейки.

# GAN_SERVER_URL = "https://xxxx-xx-xx-xxx-xx.ngrok-free.app"  # вставьте ваш URL сюда

# Тест 1: Сгенерировать одно случайное изображение
# !curl -o test_generate.png "{GAN_SERVER_URL}/generate"

# Тест 2: Сгенерировать изображение с конкретным семенем
# !curl -o test_seed.png "{GAN_SERVER_URL}/generate?seed=42"

# Тест 3: Интерполировать между двумя латентными точками
# !curl -o test_interp.png "{GAN_SERVER_URL}/interpolate?steps=8"

# Тест 4: Сгенерировать с пользовательским латентным вектором
# !curl -o test_custom.png "{GAN_SERVER_URL}/generate?z0=2.0&z1=-1.5&z2=0.5&z3=1.0"

# Отображаем сгенерированное изображение
# from PIL import Image
# import IPython.display as dp
# dp.display(dp.Image(filename='test_generate.png'))

print("Раскомментируйте команды выше и вставьте ваш ngrok URL для тестирования сервера!")

---
## Шаг 10. Условный GAN (cGAN)

cGAN (Conditional GAN) позволяет нам контролировать, ЧТО генератор создаёт, предоставляя метку класса.

### Ключевое отличие от обычного GAN

| GAN | cGAN |
|-----|------|
| G(z) -> случайное изображение | G(z, y) -> изображение класса y |
| D(x) -> реальное/фейк? | D(x, y) -> реальное/фейк для класса y? |
| Нет контроля над выходом | Управление выходом через метку |

### Архитектура cGAN для MNIST

```
z (100) + y (10, one-hot) -> [Генератор] -> 28x28 изображение цифры y
x (28x28) + y (10, one-hot) -> [Дискриминатор] -> реальное/фейк для цифры y
```

In [ ]:
# ============================================================
# ШАГ 10: Определяем модели условного GAN (cGAN)
# ============================================================

class ConditionalGenerator(nn.Module):               # Условный Генератор
    # Генерирует изображение конкретной цифры по метке класса
    def __init__(self, latent_dim=100, n_classes=10, ngf=64):  # конструктор
        super(ConditionalGenerator, self).__init__()  # вызываем родителя
        self.latent_dim = latent_dim                 # сохраняем размерность латентного пространства
        self.n_classes = n_classes                   # сохраняем количество классов

        # Эмбеддинг метки: индекс цифры -> вектор эмбеддинга
        self.label_embed = nn.Embedding(n_classes, 50)  # эмбеддинг метки класса

        # Проецируем конкатенацию (z + label_embed) в карту признаков
        self.fc = nn.Linear(latent_dim + 50, ngf * 4 * 4 * 4)  # проецируем в карту 4x4

        self.main = nn.Sequential(                   # транпонирующие свёрточные слои
            nn.BatchNorm2d(ngf * 4),                 # пакетная нормализация
            nn.ReLU(True),                           # активация

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 3, 2, 1, bias=False),  # -> 128x7x7
            nn.BatchNorm2d(ngf * 2),                 # пакетная нормализация
            nn.ReLU(True),                           # активация

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),  # -> 64x14x14
            nn.BatchNorm2d(ngf),                     # пакетная нормализация
            nn.ReLU(True),                           # активация

            nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False),  # -> 1x28x28
            nn.Tanh()                                # выход в [-1, 1]
        )

    def forward(self, z, labels):                   # прямой проход
        # Эмбеддинг меток и конкатенация с z
        label_embed = self.label_embed(labels)       # эмбеддинг меток классов
        x = torch.cat([z, label_embed], dim=1)      # конкатенация z + эмбеддинг метки
        x = self.fc(x)                              # проецируем в 4D
        x = x.view(x.size(0), -1, 4, 4)            # переформируем в (batch, channels, 4, 4)
        return self.main(x)                          # пропускаем через транпонирующие свёртки


class ConditionalDiscriminator(nn.Module):           # Условный Дискриминатор
    # Классифицирует пару изображение + метка как реальное или фейк
    def __init__(self, n_classes=10, ndf=64):       # конструктор
        super(ConditionalDiscriminator, self).__init__()  # вызываем родителя
        self.n_classes = n_classes                   # сохраняем количество классов

        # Эмбеддинг метки
        self.label_embed = nn.Embedding(n_classes, 50)  # эмбеддинг метки класса

        # Проецируем метку в пространственные размеры изображения
        self.label_fc = nn.Linear(50, 28 * 28)      # проецируем в размер изображения

        # Основные свёрточные слои (вход: 2 канала - изображение + карта метки)
        self.main = nn.Sequential(                   # свёрточные слои
            nn.Conv2d(2, ndf, 4, 2, 1, bias=False),  # -> 64x14x14
            nn.LeakyReLU(0.2, inplace=True),        # активация

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),  # -> 128x7x7
            nn.BatchNorm2d(ndf * 2),                 # пакетная нормализация
            nn.LeakyReLU(0.2, inplace=True),        # активация

            nn.Conv2d(ndf * 2, ndf * 4, 3, 2, 1, bias=False),  # -> 256x4x4
            nn.BatchNorm2d(ndf * 4),                 # пакетная нормализация
            nn.LeakyReLU(0.2, inplace=True),        # активация

            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),  # -> 1x1x1
            nn.Sigmoid()                             # вероятность
        )

    def forward(self, x, labels):                   # прямой проход
        # Эмбеддинг меток и переформирование под размер изображения
        label_embed = self.label_embed(labels)       # эмбеддинг меток
        label_map = self.label_fc(label_embed)       # проецируем в пространственные размеры
        label_map = label_map.view(x.size(0), 1, 28, 28)  # переформируем в (batch, 1, 28, 28)

        # Конкатенируем изображение с картой метки
        x_input = torch.cat([x, label_map], dim=1)  # объединяем: 2 канала
        return self.main(x_input)                    # пропускаем через сеть


# Создаём модели cGAN
G_cg = ConditionalGenerator(latent_dim=latent_dim_dc, n_classes=10).to(device)  # условный генератор
D_cg = ConditionalDiscriminator(n_classes=10).to(device)  # условный дискриминатор

# Инициализируем веса
G_cg.apply(weights_init)                            # инициализируем веса Генератора
D_cg.apply(weights_init)                            # инициализируем веса Дискриминатора

print(f"Параметры условного Генератора: {sum(p.numel() for p in G_cg.parameters()):,}")  # считаем параметры G
print(f"Параметры условного Дискриминатора: {sum(p.numel() for p in D_cg.parameters()):,}")  # считаем параметры D
print("Модели cGAN созданы!")

In [ ]:
# ============================================================
# ШАГ 10 (продолжение): Обучаем условный GAN (cGAN)
# ============================================================

# Оптимизаторы для cGAN
optimizer_G_cg = optim.Adam(G_cg.parameters(), lr=lr_dc, betas=(beta1, 0.999))  # оптимизатор G
optimizer_D_cg = optim.Adam(D_cg.parameters(), lr=lr_dc, betas=(beta1, 0.999))  # оптимизатор D

# Параметры обучения cGAN
num_epochs_cg = 15                                  # количество эпох

# Хранилище потерь
g_losses_cg = []                                    # история потерь G
d_losses_cg = []                                    # история потерь D

print(f"Начинаем обучение cGAN на {num_epochs_cg} эпох...")
print("-" * 60)

for epoch in range(1, num_epochs_cg + 1):           # цикл по эпохам
    g_loss_epoch = 0.0                              # накапливаем потерю G
    d_loss_epoch = 0.0                              # накапливаем потерю D

    for i, (real_imgs, real_labels) in enumerate(mnist_loader):  # цикл по батчам
        batch_curr = real_imgs.size(0)              # текущий размер батча
        real_imgs = real_imgs.to(device)             # переносим изображения на устройство
        real_labels = real_labels.to(device)         # переносим метки на устройство

        # Метки для обучения
        ones = torch.ones(batch_curr, 1, device=device)    # реальное = 1
        zeros = torch.zeros(batch_curr, 1, device=device)  # фейк = 0

        # --- Обучаем Дискриминатор ---
        optimizer_D_cg.zero_grad()                  # обнуляем градиенты D

        # D на реальных изображениях с правильными метками
        d_real_out = D_cg(real_imgs, real_labels)   # предсказание D на реальных
        d_loss_real = criterion_dc(d_real_out, ones)  # потеря на реальных

        # D на фейковых изображениях
        z_cg = torch.randn(batch_curr, latent_dim_dc, device=device)  # случайный шум
        fake_labels_cg = torch.randint(0, 10, (batch_curr,), device=device)  # случайные метки классов
        fake_imgs_cg = G_cg(z_cg, fake_labels_cg).detach()  # генерируем фейки (detach для D)
        d_fake_out = D_cg(fake_imgs_cg, fake_labels_cg)  # предсказание D на фейках
        d_loss_fake = criterion_dc(d_fake_out, zeros)  # потеря на фейках

        d_loss = (d_loss_real + d_loss_fake) / 2     # общая потеря D
        d_loss.backward()                            # обратное распространение
        optimizer_D_cg.step()                        # обновляем веса D

        # --- Обучаем Генератор ---
        optimizer_G_cg.zero_grad()                  # обнуляем градиенты G

        z_cg2 = torch.randn(batch_curr, latent_dim_dc, device=device)  # новый случайный шум
        fake_labels_cg2 = torch.randint(0, 10, (batch_curr,), device=device)  # новые случайные метки
        fake_imgs_cg2 = G_cg(z_cg2, fake_labels_cg2)  # генерируем фейки с градиентами
        d_fake_out2 = D_cg(fake_imgs_cg2, fake_labels_cg2)  # предсказание D на фейках G
        g_loss = criterion_dc(d_fake_out2, ones)    # G хочет чтобы D сказал "реальное"
        g_loss.backward()                            # обратное распространение
        optimizer_G_cg.step()                        # обновляем веса G

        # Накапливаем потери
        g_loss_epoch += g_loss.item()                # добавляем потерю G
        d_loss_epoch += d_loss.item()                # добавляем потерю D

    # Средние потери за эпоху
    g_losses_cg.append(g_loss_epoch / len(mnist_loader))  # средняя потеря G
    d_losses_cg.append(d_loss_epoch / len(mnist_loader))  # средняя потеря D

    # Печатаем прогресс
    print(f"Эпоха [{epoch:2d}/{num_epochs_cg}] | Потеря D: {d_losses_cg[-1]:.4f} | Потеря G: {g_losses_cg[-1]:.4f}")  # прогресс

print("\nОбучение cGAN завершено!")

In [ ]:
# ============================================================
# ШАГ 10 (продолжение): Визуализируем условную генерацию
# Генерируем каждую цифру от 0 до 9
# ============================================================

fig, axes = plt.subplots(2, 10, figsize=(20, 5))   # 2 строки, 10 столбцов (по одному на цифру)

G_cg.eval()                                         # режим оценки для Генератора

for digit in range(10):                             # для каждой цифры 0-9
    # Генерируем 2 варианта каждой цифры
    for row in range(2):                             # 2 варианта
        z_vis = torch.randn(1, latent_dim_dc, device=device)  # случайный шум
        label_vis = torch.tensor([digit], device=device)  # метка = цифра
        with torch.no_grad():                       # без градиентов
            img = G_cg(z_vis, label_vis).cpu()      # генерируем изображение цифры
        axes[row, digit].imshow(img[0, 0].numpy(), cmap='gray', vmin=-1, vmax=1)  # показываем
        axes[row, digit].axis('off')                # скрываем оси
        if row == 0:                                # заголовок только для первого ряда
            axes[row, digit].set_title(f'Цифра {digit}', fontsize=11)  # заголовок

plt.suptitle('cGAN: Условная генерация каждой цифры (0-9)', fontsize=15, fontweight='bold')  # общий заголовок
plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

---
## Шаг 11. Проблемы обучения GAN

GAN славятся своей нестабильностью обучения. Разберём основные проблемы и решения.

### 1. Mode Collapse (Коллапс мод)

**Проблема:** Генератор начинает создавать только один или несколько типов выходов, игнорируя всё разнообразие данных.

```
Реальное распределение:  /\   /\   /\   (много мод)
С Mode Collapse:         /\                  (одна мода)
```

**Решения:**
- WGAN (Wasserstein GAN) - использование расстояния Вассерштейна
- Minibatch discrimination - D смотрит на целый батч, а не отдельные примеры
- Unrolled GAN - G учитывает будущие обновления D

### 2. Нестабильность обучения

**Проблема:** Потери G и D осциллируют, не сходятся. D становится слишком сильным, G не может учиться.

**Решения:**
- Label smoothing: реальные метки = 0.9 вместо 1.0
- Нелинейное обучение D: обновляем D реже чем G
- Spectral Normalization - нормализация спектрального радиуса весов
- Progressive growing - начинаем с маленьких изображений

### 3. Исчезающие градиенты

**Проблема:** Когда D слишком хорош, градиенты G становятся нулевыми.

**Решение:** Использовать Wasserstein loss вместо оригинального GAN loss

In [ ]:
# ============================================================
# ШАГ 11: Демонстрация Mode Collapse
# Обучаем "плохой" GAN специально, чтобы показать проблему
# ============================================================

# Создаём "слабый" генератор (мало нейронов) с высоким LR
G_bad = Generator1D(latent_dim=4, hidden_dim=8).to(device)  # слабый генератор
D_bad = Discriminator1D(hidden_dim=8).to(device)  # слабый дискриминатор

opt_G_bad = optim.Adam(G_bad.parameters(), lr=0.01)  # слишком высокий LR для G
opt_D_bad = optim.Adam(D_bad.parameters(), lr=0.001)  # нормальный LR для D

bad_snapshots = {}                                  # снимки для демонстрации
bad_epochs_list = [1, 30, 100, 300]                 # эпохи для снимков

for epoch in range(1, 301):                         # цикл обучения
    real_data = generate_real_data_1d(64)           # реальные данные
    z = torch.randn(64, 4).to(device)              # латентный вектор (маленький!)

    # Обучаем D (3 раза за шаг - D слишком сильный)
    for _ in range(3):                              # 3 шага D
        opt_D_bad.zero_grad()                       # обнуляем градиенты D
        d_real = D_bad(real_data)                   # D на реальных
        d_fake = D_bad(G_bad(z).detach())           # D на фейках
        d_loss_bad = (criterion(d_real, torch.ones_like(d_real)) +
                      criterion(d_fake, torch.zeros_like(d_fake))) / 2  # потеря D
        d_loss_bad.backward()                       # обратное распространение
        opt_D_bad.step()                            # обновляем веса D

    # Обучаем G (1 раз за шаг)
    opt_G_bad.zero_grad()                           # обнуляем градиенты G
    z2 = torch.randn(64, 4).to(device)             # новый шум
    g_loss_bad = criterion(D_bad(G_bad(z2)), torch.ones(64, 1).to(device))  # потеря G
    g_loss_bad.backward()                           # обратное распространение
    opt_G_bad.step()                                # обновляем веса G

    # Сохраняем снимки
    if epoch in bad_epochs_list:                    # проверяем нужна ли эпоха
        with torch.no_grad():                       # без градиентов
            z_snap = torch.randn(512, 4).to(device)  # латентные для снимка
            bad_snapshots[epoch] = G_bad(z_snap).cpu().numpy()  # сохраняем

# Визуализируем Mode Collapse
fig, axes = plt.subplots(1, len(bad_epochs_list), figsize=(20, 4))  # подграфики
real_v = generate_real_data_1d(512).cpu().numpy()   # реальные данные

for i, ep in enumerate(bad_epochs_list):            # перебираем эпохи
    if ep in bad_snapshots:                          # проверяем наличие снимка
        fake_v = bad_snapshots[ep]                   # получаем снимок
        axes[i].hist(real_v[:, 0], bins=50, alpha=0.5, color='blue',  # реальное
                     label='Реальное', density=True)
        axes[i].hist(fake_v[:, 0], bins=50, alpha=0.5, color='red',   # фейк
                     label='Сгенерированное', density=True)
        axes[i].set_title(f'Эпоха {ep}', fontsize=12)  # заголовок
        axes[i].legend(fontsize=9)                   # легенда

plt.suptitle('Mode Collapse: Генератор создаёт только одно значение!', fontsize=14, fontweight='bold')  # заголовок
plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

---
## Шаг 11 (бонус). Концепция StyleGAN

StyleGAN (Karras et al., 2019) - это революционная архитектура GAN для генерации фотореалистичных изображений.

### Ключевые идеи StyleGAN

| Идея | Описание |
|------|----------|
| **Прогрессивный рост** | Начинаем с 4x4, постепенно добавляем слои до 1024x1024 |
| **Mapping network** | z -> w через 8-слойный MLP (промежуточное латентное пространство) |
| **AdaIN** | Adaptive Instance Normalization - инъекция стиля на каждом слое |
| **Stochastic variation** | Случайный шум для мелких деталей (веснушки, волосы) |
| **Style mixing** | Разные стили на разных слоях = контроль грубых/мелких деталей |

### Архитектура StyleGAN

```
z -> [Mapping Network f] -> w (промежуточный стиль)
                              |
                              v
z_noise -> [Synthesis Network g] -> изображение
            |    |    |    |
           AdaIN AdaIN AdaIN AdaIN  <- w подаётся на каждый слой
            |    |    |    |
          шум  шум  шум  шум       <- стохастическая вариация
```

### Слои стиля

| Слой | Что контролирует | Пример |
|------|-----------------|--------|
| Грубый (4x4 - 8x8) | Поза, форма лица | Овальное/круглое лицо |
| Средний (16x16 - 32x32) | Черты лица | Широкий/узкий нос |
| Мелкий (64x64 - 1024x1024) | Цвет, текстура | Цвет глаз, кожа

In [ ]:
# ============================================================
# ШАГ 11 (бонус): Визуализируем концепцию StyleGAN
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(16, 8))      # создаём фигуру

# Mapping Network
mapping_box = plt.Rectangle((0.02, 0.35), 0.18, 0.3,  # Mapping Network
                             facecolor='#4CAF50', alpha=0.8, edgecolor='black', linewidth=2)
ax.add_patch(mapping_box)
ax.text(0.11, 0.5, 'Mapping\nNetwork\nf: z -> w', ha='center', va='center',
        fontsize=11, fontweight='bold', color='white')

# Слои Synthesis Network
layer_names = ['4x4', '8x8', '16x16', '32x32', '64x64']  # разрешения слоёв
style_types = ['Грубый', 'Грубый', 'Средний', 'Средний', 'Мелкий']  # типы стиля
colors = ['#F44336', '#F44336', '#FF9800', '#FF9800', '#4CAF50']  # цвета по типу стиля

for i, (name, stype, color) in enumerate(zip(layer_names, style_types, colors)):  # перебираем слои
    x_pos = 0.28 + i * 0.14                        # позиция X блока
    rect = plt.Rectangle((x_pos, 0.35), 0.11, 0.3,  # блок слоя
                         facecolor=color, alpha=0.7, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x_pos + 0.055, 0.52, f'{name}\n{stype}',  # подпись слоя
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')

    # AdaIN стрелка (сверху)
    ax.annotate('AdaIN', xy=(x_pos + 0.055, 0.65), xytext=(0.11, 0.85),  # от Mapping к слою
                arrowprops=dict(arrowstyle='->', lw=1.5, color='blue'),
                fontsize=8, color='blue', ha='center')

    # Шум (снизу)
    ax.annotate(f'шум', xy=(x_pos + 0.055, 0.35), xytext=(x_pos + 0.055, 0.15),  # шум к слою
                arrowprops=dict(arrowstyle='->', lw=1.2, color='gray'),
                fontsize=8, color='gray', ha='center')

    # Стрелка между слоями
    if i > 0:                                       # стрелка от предыдущего слоя
        ax.annotate('', xy=(x_pos, 0.5), xytext=(x_pos - 0.03, 0.5),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))

# Вход z
ax.text(0.02, 0.75, 'z (латентный)', fontsize=11, color='blue', fontweight='bold')
ax.annotate('', xy=(0.11, 0.65), xytext=(0.11, 0.73),
            arrowprops=dict(arrowstyle='->', lw=2, color='blue'))

# Выход
ax.text(0.92, 0.5, '-> 1024x1024\nизображение', fontsize=11, fontweight='bold', ha='left', va='center')

ax.set_xlim(0, 1.15)                                # пределы оси X
ax.set_ylim(0.05, 0.95)                             # пределы оси Y
ax.set_title('Концепция StyleGAN: инъекция стиля через AdaIN', fontsize=16, fontweight='bold')  # заголовок
ax.axis('off')                                       # скрываем оси

plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

---
## Шаг 12. Не только GAN: VAE, Diffusion Models, Flow

GAN - не единственная генеративная модель. Сравним основные подходы.

### Сравнение генеративных моделей

| Характеристика | GAN | VAE | Diffusion | Flow |
|----------------|-----|-----|-----------|------|
| **Обучение** | Состязательное | Реконструкция + KL | Шумоочистка | Правдоподобие |
| **Качество выборок** | Высокое | Среднее | Очень высокое | Среднее-высокое |
| **Стабильность обучения** | Нестабильное | Стабильное | Стабильное | Стабильное |
| **Правдоподобие** | Нет | Нижняя граница | Приближённое | Точное |
| **Разнообразие** | Риск mode collapse | Хорошее | Отличное | Хорошее |
| **Скорость генерации** | Быстрая | Быстрая | Медленная (много шагов) | Быстрая |
| **Латентное пространство** | Неструктурированное | Структурированное | Через guidance | Структурированное |
| **Ключевые статьи** | Goodfellow 2014 | Kingma 2013 | Ho 2020 | Dinh 2014 |

In [ ]:
# ============================================================
# ШАГ 12: Визуальное сравнение семейств генеративных моделей
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(14, 8))      # одна фигура

# Метрики сравнения для каждого семейства моделей
models = ['GAN', 'VAE', 'Diffusion', 'Flow']       # названия моделей
metrics = {                                         # метрики для радарной диаграммы
    'Качество выборок': [9, 6, 9.5, 7],            # визуальное качество выборок
    'Стабильность обучения': [4, 8, 8, 7],         # лёгкость обучения
    'Разнообразие': [5, 8, 9, 8],                  # покрытие мод
    'Скорость генерации': [9, 9, 2, 8],            # скорость генерации
    'Правдоподобие': [2, 7, 7, 9],                 # точное/приближённое правдоподобие
    'Управляемость': [6, 8, 9, 7],                 # контроль над генерацией
}

# Количество метрик
n_metrics = len(metrics)                            # считаем метрики
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()  # углы для радара
angles += angles[:1]                                # замыкаем полигон

# Цвета для каждой модели
colors = ['#F44336', '#4CAF50', '#2196F3', '#FF9800']  # красный, зелёный, синий, оранжевый

# Строим каждую модель
for i, model in enumerate(models):                  # для каждой модели
    values = [metrics[m][i] for m in metrics]       # получаем значения метрик
    values += values[:1]                             # замыкаем полигон
    ax.plot(angles, values, 'o-', linewidth=2, color=colors[i], label=model, markersize=8)  # строим линию
    ax.fill(angles, values, alpha=0.1, color=colors[i])  # заливаем область

# Настраиваем радарную диаграмму
ax.set_xticks(angles[:-1])                          # устанавливаем позиции делений
ax.set_xticklabels(list(metrics.keys()), fontsize=11)  # подписи метрик
ax.set_ylim(0, 10)                                  # пределы оси Y
ax.set_yticks([2, 4, 6, 8, 10])                    # деления оси Y
ax.set_yticklabels(['2', '4', '6', '8', '10'], fontsize=9)  # подписи делений Y
ax.set_title('Сравнение генеративных моделей', fontsize=16, fontweight='bold', pad=20)  # заголовок
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12)  # легенда

plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

In [ ]:
# ============================================================
# ШАГ 12 (продолжение): Архитектурные диаграммы VAE, Diffusion, Flow
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 5))    # 3 подграфика

# --- Архитектура VAE ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
# Энкодер
rect = plt.Rectangle((0.5, 1.5), 2, 2, facecolor='#4CAF50', alpha=0.7, edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(1.5, 2.5, 'Энкодер\nq(z|x)', ha='center', va='center', fontsize=11, fontweight='bold')
# Латентное пространство
rect = plt.Rectangle((3.5, 1.5), 2, 2, facecolor='#FFC107', alpha=0.7, edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(4.5, 2.5, 'z (mu, sigma)\nрепараметризация', ha='center', va='center', fontsize=10, fontweight='bold')
# Декодер
rect = plt.Rectangle((6.5, 1.5), 2, 2, facecolor='#2196F3', alpha=0.7, edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(7.5, 2.5, 'Декодер\np(x|z)', ha='center', va='center', fontsize=11, fontweight='bold')
# Стрелки
ax.annotate('', xy=(3.5, 2.5), xytext=(2.5, 2.5), arrowprops=dict(arrowstyle='->', lw=2))
ax.annotate('', xy=(6.5, 2.5), xytext=(5.5, 2.5), arrowprops=dict(arrowstyle='->', lw=2))
ax.set_title('VAE: Кодирование -> Выборка -> Декодирование', fontsize=13, fontweight='bold')
ax.axis('off')

# --- Diffusion Model ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
# Чистое изображение
rect = plt.Rectangle((0.5, 1.5), 2, 2, facecolor='#4CAF50', alpha=0.7, edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(1.5, 2.5, 'x_0\n(чистое)', ha='center', va='center', fontsize=11, fontweight='bold')
# Зашумлённое
rect = plt.Rectangle((3.5, 1.5), 2, 2, facecolor='#9E9E9E', alpha=0.7, edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(4.5, 2.5, 'x_t\n(шум)', ha='center', va='center', fontsize=11, fontweight='bold')
# Чистый шум
rect = plt.Rectangle((6.5, 1.5), 2, 2, facecolor='#212121', alpha=0.7, edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(7.5, 2.5, 'x_T\n(шум)', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
# Стрелки прямого процесса
ax.annotate('', xy=(3.5, 2.5), xytext=(2.5, 2.5), arrowprops=dict(arrowstyle='->', lw=2, color='red'))
ax.text(3.0, 3.2, 'добавляем\nшум', fontsize=9, color='red', ha='center')
ax.annotate('', xy=(6.5, 2.5), xytext=(5.5, 2.5), arrowprops=dict(arrowstyle='->', lw=2, color='red'))
# Обратная стрелка (снизу)
ax.annotate('', xy=(0.5, 1.0), xytext=(7.5, 1.0),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='green', connectionstyle='arc3,rad=-0.2'))
ax.text(4.0, 0.2, 'Шумоочистка (обратный процесс)', fontsize=10, color='green', ha='center', fontweight='bold')
ax.set_title('Diffusion: Шум -> Очистка -> Чистое', fontsize=13, fontweight='bold')
ax.axis('off')

# --- Flow Model ---
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
# Пространство данных
rect = plt.Rectangle((0.5, 1.5), 2, 2, facecolor='#4CAF50', alpha=0.7, edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(1.5, 2.5, 'x\n(данные)', ha='center', va='center', fontsize=11, fontweight='bold')
# Преобразование
rect = plt.Rectangle((3.5, 1.5), 2, 2, facecolor='#FF9800', alpha=0.7, edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(4.5, 2.5, 'f(x)\n(биекция)', ha='center', va='center', fontsize=11, fontweight='bold')
# Латентное пространство
rect = plt.Rectangle((6.5, 1.5), 2, 2, facecolor='#2196F3', alpha=0.7, edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(7.5, 2.5, 'z\n(латентное)', ha='center', va='center', fontsize=11, fontweight='bold')
# Стрелки прямого преобразования
ax.annotate('', xy=(3.5, 3.0), xytext=(2.5, 3.0), arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
ax.text(3.0, 3.5, 'f', fontsize=12, color='blue', ha='center', fontweight='bold')
ax.annotate('', xy=(6.5, 2.0), xytext=(5.5, 2.0), arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
# Стрелка обратного преобразования
ax.annotate('', xy=(2.5, 2.0), xytext=(3.5, 2.0), arrowprops=dict(arrowstyle='->', lw=2, color='red'))
ax.text(3.0, 1.3, 'f^(-1)', fontsize=12, color='red', ha='center', fontweight='bold')
ax.set_title('Flow: x <-> z (обратимое)', fontsize=13, fontweight='bold')
ax.axis('off')

plt.suptitle('Архитектуры генеративных моделей', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ШАГ 12 (продолжение): Реализуем простой VAE для сравнения
# ============================================================

class SimpleVAE(nn.Module):                         # Простой Вариационный Автоэнкодер
    # VAE: Энкодер -> (mu, log_var) -> репараметризация -> z -> Декодер
    def __init__(self, latent_dim=20):              # конструктор
        super(SimpleVAE, self).__init__()           # вызываем родителя
        self.latent_dim = latent_dim                 # сохраняем размерность латентного пространства

        # Энкодер: изображение -> mu и log_var
        self.encoder = nn.Sequential(               # сеть энкодера
            nn.Linear(28 * 28, 512),                # выравниваем изображение -> 512
            nn.ReLU(),                              # активация
            nn.Linear(512, 256),                    # 512 -> 256
            nn.ReLU(),                              # активация
        )
        self.fc_mu = nn.Linear(256, latent_dim)     # 256 -> среднее латентного
        self.fc_logvar = nn.Linear(256, latent_dim)  # 256 -> лог-дисперсия латентного

        # Декодер: z -> реконструированное изображение
        self.decoder = nn.Sequential(               # сеть декодера
            nn.Linear(latent_dim, 256),             # латентное -> 256
            nn.ReLU(),                              # активация
            nn.Linear(256, 512),                    # 256 -> 512
            nn.ReLU(),                              # активация
            nn.Linear(512, 28 * 28),                # 512 -> пиксели изображения
            nn.Sigmoid(),                           # выход [0, 1]
        )

    def encode(self, x):                           # кодируем изображение в (mu, log_var)
        h = self.encoder(x.view(-1, 28 * 28))       # выравниваем и кодируем
        return self.fc_mu(h), self.fc_logvar(h)     # возвращаем mu и log_var

    def reparameterize(self, mu, logvar):           # трюк репараметризации
        std = torch.exp(0.5 * logvar)               # вычисляем std из log_var
        eps = torch.randn_like(std)                  # семплируем эпсилон
        return mu + eps * std                        # z = mu + eps * sigma

    def decode(self, z):                            # декодируем латентное в изображение
        return self.decoder(z)                       # пропускаем через декодер

    def forward(self, x):                           # полный прямой проход
        mu, logvar = self.encode(x)                  # кодируем
        z = self.reparameterize(mu, logvar)          # семплируем z
        recon = self.decode(z)                       # декодируем
        return recon, mu, logvar                     # возвращаем реконструкцию + параметры латентного


def vae_loss(recon_x, x, mu, logvar):              # функция потерь VAE
    # Потеря реконструкции (бинарная кросс-энтропия)
    BCE = F.binary_cross_entropy(recon_x, x.view(-1, 28 * 28), reduction='sum')  # реконструкция
    # KL-дивергенция: D_KL(q(z|x) || p(z))
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())  # KL-дивергенция
    return BCE + KLD                                 # общая потеря


# Создаём и тестируем VAE
vae = SimpleVAE(latent_dim=20).to(device)           # создаём VAE
print(f"Параметры VAE: {sum(p.numel() for p in vae.parameters()):,}")  # считаем параметры

# Быстро обучаем VAE на нескольких эпохах
optimizer_vae = optim.Adam(vae.parameters(), lr=1e-3)  # оптимизатор VAE
vae.train()                                         # режим обучения

print("Обучаем VAE 5 эпох...")
for epoch in range(1, 6):                           # 5 эпох
    train_loss = 0                                  # накапливаем потерю
    for batch_idx, (data, _) in enumerate(mnist_loader):  # цикл по батчам
        data = data.to(device)                       # переносим на устройство
        optimizer_vae.zero_grad()                   # обнуляем градиенты
        recon, mu, logvar = vae(data)               # прямой проход
        loss = vae_loss(recon, data, mu, logvar)    # вычисляем потерю
        loss.backward()                              # обратное распространение
        train_loss += loss.item()                    # накапливаем потерю
        optimizer_vae.step()                         # обновляем веса
    print(f"  Эпоха {epoch}: Потеря = {train_loss / len(mnist_loader.dataset):.4f}")  # прогресс

print("Обучение VAE завершено!")

In [ ]:
# ============================================================
# ШАГ 12 (продолжение): Сравниваем GAN vs VAE сгенерированные изображения
# ============================================================

fig, axes = plt.subplots(2, 8, figsize=(20, 5))    # 2 ряда: GAN и VAE

# Ряд 1: DCGAN сгенерированные изображения
for i in range(8):                                  # 8 примеров
    z = torch.randn(1, latent_dim_dc, device=device)  # случайный шум
    with torch.no_grad():                           # без градиентов
        img_gan = G_dc(z).cpu()                     # GAN генерирует
    axes[0, i].imshow(img_gan[0, 0].numpy(), cmap='gray', vmin=-1, vmax=1)  # показываем
    axes[0, i].axis('off')                         # скрываем оси

# Ряд 2: VAE сгенерированные изображения
vae.eval()                                          # режим оценки
for i in range(8):                                  # 8 примеров
    z_vae = torch.randn(1, 20).to(device)           # случайное латентное для VAE
    with torch.no_grad():                           # без градиентов
        img_vae = vae.decode(z_vae).cpu()           # VAE декодирует
    img_vae = img_vae.view(28, 28).numpy()          # переформируем в 28x28
    axes[1, i].imshow(img_vae, cmap='gray')         # показываем
    axes[1, i].axis('off')                         # скрываем оси

axes[0, 0].set_ylabel('DCGAN', fontsize=14, rotation=0, labelpad=50, fontweight='bold')  # подпись
axes[1, 0].set_ylabel('VAE', fontsize=14, rotation=0, labelpad=50, fontweight='bold')    # подпись

plt.suptitle('GAN vs VAE: Сравнение качества сгенерированных изображений', fontsize=15, fontweight='bold')  # заголовок
plt.tight_layout()                                   # подгоняем макет
plt.show()                                           # отображаем фигуру

---
## Шаг 13. Практические задания

Выполните эти упражнения для закрепления понимания GAN.

### Задание 1: Эксперимент с архитектурой 1D GAN
- Измените размер скрытого слоя с 32 на 64, 128
- Наблюдайте, как меняется динамика обучения
- Попробуйте разные функции активации (ReLU vs LeakyReLU vs ELU)

### Задание 2: Реализуйте WGAN-GP
- Замените стандартную потерю GAN на потерю Вассерштейна
- Добавьте штраф градиента: ограничение 1-Липшиц
- Сравните стабильность обучения со стандартным GAN

### Задание 3: Постройте Progressive GAN
- Начните с изображений 7x7, постепенно увеличьте до 28x28
- Добавляйте новые слои во время обучения (растите сеть)
- Визуализируйте прогрессию качества изображений

### Задание 4: API условной генерации изображений
- Расширьте FastAPI сервер эндпоинтом /generate_digit
- Принимайте метку цифры (0-9) как параметр
- Возвращайте сгенерированное изображение конкретной цифры
- Добавьте пакетную генерацию для всех 10 цифр

### Задание 5: Метрики оценки GAN
- Реализуйте Inception Score (IS)
- Реализуйте Frechet Inception Distance (FID)
- Оцените ваши модели DCGAN и cGAN
- Сравните метрики и интерпретируйте результаты

---

### Итоги

В этом блокноте вы:
1. Поняли фреймворк GAN: состязательная игра Генератор vs Дискриминатор
2. Разобрали минимаксную потерю и равновесие Нэша
3. Построили простой GAN с нуля для 1D данных
4. Визуализировали, как GAN учится совпадать с распределениями
5. Реализовали DCGAN для генерации изображений MNIST
6. Исследовали латентное пространство интерактивно
7. Создали FastAPI сервер для генерации изображений
8. Построили условный GAN для генерации конкретных цифр
9. Диагностировали mode collapse и нестабильность обучения
10. Изучили концепцию StyleGAN
11. Сравнили GAN с VAE, Diffusion и Flow моделями

> Помните: вы - автор, не потребитель. Ломайте, чините, понимайте. Никаких чёрных ящиков!